# Chapter 2: Sampling

Chapter 1 built manual autoregressive inference. This chapter replaces greedy `argmax` selection with an actual sampling pipeline, built without `generate()`.

In [1]:
!pip install -q transformers accelerate


In [2]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DynamicCache,
)

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda",
)

model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [4]:
print(model.device)
print(model.dtype)
print(model.config.vocab_size)

cuda:0
torch.float16
151936


In [5]:
prompt = "The fastest animal in the world is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

input_ids = inputs["input_ids"]

print(input_ids)
print(input_ids.shape)

tensor([[  785, 25648,  9864,   304,   279,  1879,   374]], device='cuda:0')
torch.Size([1, 7])


## Token Sampling

In our previous manual decoding loop, we selected the next token using **greedy decoding**:

```python
next_token = torch.argmax(logits[:, -1, :], dim=-1)
```

Greedy decoding always chooses the token with the highest logit. While simple and deterministic, it ignores the rest of the model's predicted token distribution.

A causal language model does not directly output a single next token. Instead, for every decoding step, it produces a **logit for every token in the vocabulary**.

For Qwen2.5-0.5B-Instruct:

```text
Vocabulary size = 151,936 tokens
```

Therefore, for a single sequence, the model produces approximately:

```text
151,936 candidate scores
```

for the next token.

These raw scores are called **logits**.

Logits are converted into probabilities using the softmax function:

```text
logits
   ↓
softmax
   ↓
probability distribution
```

Instead of always choosing the highest-probability token, **sampling** draws a token according to this probability distribution.

The decoding pipeline becomes:

```text
model forward pass
        ↓
next-token logits
        ↓
temperature scaling
        ↓
softmax
        ↓
optional top-k / top-p filtering
        ↓
sample next token
```

Sampling allows multiple valid continuations to be generated from the same prompt and forms the basis of commonly used decoding controls such as **temperature, top-k, and top-p sampling**.


In [6]:
with torch.no_grad():
    outputs = model(input_ids)

print(outputs.logits.shape)

torch.Size([1, 7, 151936])


In [7]:
last_token_logits = outputs.logits[:, -1, :]

print(last_token_logits.shape)

torch.Size([1, 151936])


In [8]:
#applying topk method

top_values, top_indices = torch.topk(last_token_logits, k=10)

for score, token_id in zip(top_values[0], top_indices[0]):
    token = tokenizer.decode([token_id.item()])
    print(f"{token_id.item():6d} | {score.item():8.3f} | {repr(token)}")

   279 |   15.477 | ' the'
   264 |   14.617 | ' a'
  4363 |   13.102 | ' likely'
  4658 |   12.930 | ' probably'
  1304 |   12.445 | ' __'
   537 |   12.438 | ' not'
   510 |   12.242 | ':\n'
 32671 |   12.219 | ' ______'
   458 |   12.117 | ' an'
  1083 |   12.039 | ' also'


## Inspecting Next-Token Logits

Before implementing sampling, we first inspect the raw logits produced by the model.

```python
with torch.no_grad():
    outputs = model(input_ids)

print(outputs.logits.shape)

last_token_logits = outputs.logits[:, -1, :]

print(last_token_logits.shape)
```

Output:

```text
torch.Size([1, 7, 151936])
torch.Size([1, 151936])
```

The full logits tensor has the shape:

```text
[batch_size, sequence_length, vocabulary_size]

[1, 7, 151936]
```

Our prompt contains 7 tokens. For every token position, the model produces one logit for each of the 151,936 tokens in its vocabulary.

For next-token generation, we only need the logits from the final sequence position:

```python
last_token_logits = outputs.logits[:, -1, :]
```

Here:

```text
:   → all items in the batch
-1  → final token position in the sequence
:   → all vocabulary logits
```

This reduces the tensor from:

```text
[1, 7, 151936]
```

to:

```text
[1, 151936]
```

The resulting tensor contains the model's scores for every possible token that could come next.

### Inspecting the Highest-Scoring Tokens

We can inspect the model's top candidates using `torch.topk`:

```python
top_values, top_indices = torch.topk(last_token_logits, k=10)

for score, token_id in zip(top_values[0], top_indices[0]):
    token = tokenizer.decode([token_id.item()])
    print(f"{token_id.item():6d} | {score.item():8.3f} | {repr(token)}")
```

Output:

```text
   279 |   15.477 | ' the'
   264 |   14.617 | ' a'
  4363 |   13.102 | ' likely'
  4658 |   12.930 | ' probably'
  1304 |   12.445 | ' __'
   537 |   12.438 | ' not'
   510 |   12.242 | ':\n'
 32671 |   12.219 | ' ______'
   458 |   12.117 | ' an'
  1083 |   12.039 | ' also'
```

`torch.topk(..., k=10)` returns:

```text
top_values   → the 10 largest logits
top_indices  → the vocabulary indices where those logits occur
```

Because the final tensor dimension corresponds directly to the vocabulary, these indices are also the token IDs.

For example:

```text
token ID 279 → ' the'
```

The values such as `15.477` and `14.617` are still **logits**, not probabilities. A higher logit means the model considers that token more likely relative to other candidates, but the value itself is not a percentage.

The next step is to convert these logits into a probability distribution using **softmax**.


In [9]:
probs = torch.softmax(last_token_logits, dim=-1)

print(probs.shape)
print(probs.sum()) #sum of all probabilities should be 1 (here its approximately one )
# here we got 0.9995 instead of  1.0 because of FP16 rounding error. we need higher precision

torch.Size([1, 151936])
tensor(0.9995, device='cuda:0', dtype=torch.float16)


In [10]:
#inspecting actual probabilities of top k tokens predicted

top_probs, top_indices = torch.topk(probs, k=10)

for prob, token_id in zip(top_probs[0], top_indices[0]):
    token = tokenizer.decode([token_id.item()])
    print(
        f"{token_id.item():6d} | "
        f"{prob.item():.4f} | "
        f"{prob.item() * 100:.2f}% | "
        f"{repr(token)}"
    )

   279 | 0.4324 | 43.24% | ' the'
   264 | 0.1831 | 18.31% | ' a'
  4363 | 0.0402 | 4.02% | ' likely'
  4658 | 0.0339 | 3.39% | ' probably'
  1304 | 0.0209 | 2.09% | ' __'
   537 | 0.0207 | 2.07% | ' not'
   510 | 0.0170 | 1.70% | ':\n'
 32671 | 0.0166 | 1.66% | ' ______'
   458 | 0.0150 | 1.50% | ' an'
  1083 | 0.0139 | 1.39% | ' also'


## Converting Logits to Probabilities with Softmax

The model's output logits are raw relative scores. To use them for sampling, we first convert them into a probability distribution using **softmax**.

```python
probs = torch.softmax(last_token_logits, dim=-1)

print(probs.shape)
print(probs.sum())
```

Output:

```text
torch.Size([1, 151936])
tensor(0.9995, device='cuda:0', dtype=torch.float16)
```

The tensor shape remains:

```text
[batch_size, vocabulary_size]
[1, 151936]
```

so every vocabulary token now has a corresponding probability.

`dim=-1` tells PyTorch to apply softmax across the final dimension, which in this case is the vocabulary dimension containing all 151,936 possible next tokens.

Conceptually:

```text
151,936 logits
      ↓
    softmax
      ↓
151,936 probabilities
```

Softmax converts each logit \(z_i\) into a probability:

$$
P_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

After softmax:

* every value is between 0 and 1
* higher logits still correspond to higher probabilities
* all token probabilities sum to approximately 1

The observed sum is `0.9995` rather than exactly `1.0` because the tensor is stored in `float16`. FP16 has limited numerical precision, so summing a very large number of small values introduces minor rounding error.

### Inspecting the Probability Distribution

We can inspect the highest-probability next-token candidates:

```python
top_probs, top_indices = torch.topk(probs, k=10)

for prob, token_id in zip(top_probs[0], top_indices[0]):
    token = tokenizer.decode([token_id.item()])

    print(
        f"{token_id.item():6d} | "
        f"{prob.item():.4f} | "
        f"{prob.item() * 100:.2f}% | "
        f"{repr(token)}"
    )
```

Output:

```text
   279 | 0.4324 | 43.24% | ' the'
   264 | 0.1831 | 18.31% | ' a'
  4363 | 0.0402 | 4.02%  | ' likely'
  4658 | 0.0339 | 3.39%  | ' probably'
  1304 | 0.0209 | 2.09%  | ' __'
   537 | 0.0207 | 2.07%  | ' not'
   510 | 0.0170 | 1.70%  | ':\n'
 32671 | 0.0166 | 1.66%  | ' ______'
   458 | 0.0150 | 1.50%  | ' an'
  1083 | 0.0139 | 1.39%  | ' also'
```

The highest-scoring token, `' the'`, now has a probability of approximately **43.24%**, while `' a'` has approximately **18.31%**.

These probabilities can now be used to **sample** a next token rather than always selecting the maximum-probability token with `argmax`.

Softmax preserves the ranking of the logits:

```text
higher logit
    ↓
higher probability
```

It changes the raw scores into a normalized probability distribution without changing which tokens are ranked above others.


In [11]:
next_token= torch.multinomial(probs,num_samples=1)
print(next_token)
print(next_token.shape)
print(tokenizer.decode(next_token[0]))


tensor([[279]], device='cuda:0')
torch.Size([1, 1])
 the


## Sampling a Token

Now that we have a probability distribution over the vocabulary, we can sample the next token.

```python
next_token = torch.multinomial(probs, num_samples=1)

print(next_token)
print(next_token.shape)
print(tokenizer.decode(next_token[0]))
```

Output:

```text
tensor([[264]], device='cuda:0')
torch.Size([1, 1])
 a
```

The key operation is:

```python
torch.multinomial(probs, num_samples=1)
```

`torch.multinomial` samples an index according to the probabilities in `probs`.

In our distribution, some of the highest-probability tokens were:

```text
'the' → 43.24%
'a'   → 18.31%
...
```

Unlike greedy decoding, sampling does not always choose the highest-probability token.

Instead, every token has a chance of being selected based on its probability.

Conceptually:

```text
probability distribution
        ↓
weighted random draw
        ↓
one sampled token ID
```

In this run, token ID `264` was selected:

```text
264 → ' a'
```

Even though `' the'` had the highest probability, `' a'` still had an 18.31% probability and was selected during this particular draw.

### Output Shape

The probability tensor has shape:

```text
[1, 151936]
```

meaning:

```text
1 probability distribution
151,936 possible tokens
```

With:

```python
num_samples=1
```

we request one sampled token from each distribution.

Therefore the output shape becomes:

```text
[1, 1]
```

The first dimension represents the batch, while the second contains the sampled token ID.

This is the fundamental difference between greedy decoding and sampling:

```text
Greedy decoding:
probabilities → highest probability token

Sampling:
probabilities → weighted random token
```

Sampling allows the same model state to produce different valid continuations across different generation runs.


In [12]:
#concept of temperature in sampling (Scaling by n)

for temperature in [0.5, 1.0, 1.5]:
    scaled_logits = last_token_logits / temperature
    temp_probs = torch.softmax(scaled_logits, dim=-1)

    top_probs, top_indices = torch.topk(temp_probs, k=5)

    print(f"\nTemperature = {temperature}")

    for prob, token_id in zip(top_probs[0], top_indices[0]):
        token = tokenizer.decode([token_id.item()])
        print(
            f"{repr(token):12s} | "
            f"{prob.item() * 100:.2f}%"
        )


Temperature = 0.5
' the'       | 82.81%
' a'         | 14.84%
' likely'    | 0.72%
' probably'  | 0.51%
' __'        | 0.19%

Temperature = 1.0
' the'       | 43.24%
' a'         | 18.31%
' likely'    | 4.02%
' probably'  | 3.39%
' __'        | 2.09%

Temperature = 1.5
' the'       | 10.75%
' a'         | 6.03%
' likely'    | 2.20%
' probably'  | 1.96%
' __'        | 1.42%


## Temperature Sampling

Basic sampling uses the probability distribution produced by softmax and randomly draws the next token.

However, we often want control over **how concentrated or how spread out that probability distribution is**.

This is what **temperature** does.

Temperature is applied directly to the logits before softmax:

```python
scaled_logits = last_token_logits / temperature
probs = torch.softmax(scaled_logits, dim=-1)
```

Mathematically:

$$
P_i =
\frac{
e^{z_i/T}
}{
\sum_j e^{z_j/T}
}
$$

where:

* \(z_i\) is the original logit for token \(i\)
* \(T\) is the temperature
* \(P_i\) is the resulting probability

The important operation is:

```text
logit / temperature
```

Temperature does not change the ranking of the tokens. Instead, it changes the **distance between their effective logits**, which changes how strongly softmax favors the highest-scoring tokens.

---

### Temperature = 1.0

When:

```text
T = 1.0
```

the logits are unchanged:

```text
logit / 1.0 = logit
```

Therefore, this produces the model's original probability distribution.

For our prompt:

```text
Temperature = 1.0

' the'       | 43.24%
' a'         | 18.31%
' likely'    | 4.02%
' probably'  | 3.39%
' __'        | 2.09%
```

This is our baseline distribution.

---

### Temperature < 1: Sharper Distribution

We tested:

```text
T = 0.5
```

Dividing by a number smaller than 1 increases the magnitude of the logits.

For example:

```text
original logits:

10
8
6
```

At `T = 0.5`:

```text
10 / 0.5 = 20
8  / 0.5 = 16
6  / 0.5 = 12
```

The ranking has not changed:

```text
20 > 16 > 12
```

but the differences between the logits have become much larger.

Softmax uses exponentials, so increasing these differences makes the highest-scoring tokens dominate the probability distribution.

Our model produced:

```text
Temperature = 0.5

' the'       | 82.81%
' a'         | 14.84%
' likely'    | 0.72%
' probably'  | 0.51%
' __'        | 0.19%
```

Compare this with the original distribution:

```text
               T = 0.5      T = 1.0

' the'          82.81%       43.24%
' a'            14.84%       18.31%
' likely'        0.72%        4.02%
```

At lower temperature, most of the probability mass moves toward the highest-scoring tokens.

Conceptually:

```text
lower temperature
        ↓
larger effective logit differences
        ↓
sharper softmax distribution
        ↓
high-probability tokens dominate
        ↓
less randomness
```

Generation therefore becomes more predictable and conservative.

---

### Temperature > 1: Flatter Distribution

We also tested:

```text
T = 1.5
```

Dividing by a number greater than 1 reduces the magnitude of the logits.

For example:

```text
original:

10
8
6
```

At `T = 2`:

```text
10 / 2 = 5
8  / 2 = 4
6  / 2 = 3
```

The logits are now closer together.

Again, their ranking is unchanged:

```text
5 > 4 > 3
```

but softmax now sees a smaller difference between the candidates.

Our model produced:

```text
Temperature = 1.5

' the'       | 10.75%
' a'         | 6.03%
' likely'    | 2.20%
' probably'  | 1.96%
' __'        | 1.42%
```

The highest-probability token dropped dramatically:

```text
T = 0.5 → ' the' = 82.81%

T = 1.0 → ' the' = 43.24%

T = 1.5 → ' the' = 10.75%
```

The probability did not disappear. Instead, much more probability mass was distributed across the rest of the 151,936-token vocabulary.

Conceptually:

```text
higher temperature
        ↓
smaller effective logit differences
        ↓
flatter probability distribution
        ↓
lower-ranked tokens receive more probability
        ↓
more randomness
```

---

## Visual Intuition

Temperature can be thought of as controlling how strongly the model trusts its own ranking.

```text
Low temperature

Token A  ████████████████████████████████
Token B  ██████
Token C  █
Token D
Token E

        very concentrated
```

```text
High temperature

Token A  ████████
Token B  █████
Token C  ████
Token D  ███
Token E  ██

        more spread out
```

The token ranking remains the same, but their relative probabilities change.

---

## Why Temperature Must Be Applied Before Softmax

The correct order is:

```text
logits
   ↓
divide by temperature
   ↓
softmax
   ↓
probabilities
   ↓
sample
```

or:

```python
scaled_logits = last_token_logits / temperature
probs = torch.softmax(scaled_logits, dim=-1)

next_token = torch.multinomial(
    probs,
    num_samples=1
)
```

Temperature is fundamentally modifying the **relative logit differences that softmax receives**.

It should therefore be applied before converting the logits into probabilities.

---

## Experiment

We compared three temperatures:

```python
for temperature in [0.5, 1.0, 1.5]:
    scaled_logits = last_token_logits / temperature
    temp_probs = torch.softmax(scaled_logits, dim=-1)

    top_probs, top_indices = torch.topk(
        temp_probs,
        k=5
    )

    print(f"\nTemperature = {temperature}")

    for prob, token_id in zip(
        top_probs[0],
        top_indices[0]
    ):
        token = tokenizer.decode([token_id.item()])

        print(
            f"{repr(token):12s} | "
            f"{prob.item() * 100:.2f}%"
        )
```

Observed output:

```text
Temperature = 0.5

' the'       | 82.81%
' a'         | 14.84%
' likely'    | 0.72%
' probably'  | 0.51%
' __'        | 0.19%


Temperature = 1.0

' the'       | 43.24%
' a'         | 18.31%
' likely'    | 4.02%
' probably'  | 3.39%
' __'        | 2.09%


Temperature = 1.5

' the'       | 10.75%
' a'         | 6.03%
' likely'    | 2.20%
' probably'  | 1.96%
' __'        | 1.42%
```

The experiment clearly shows that temperature controls the **shape of the probability distribution**, rather than directly deciding which token should be generated.

The actual token is still selected later through sampling.

```text
T < 1  → sharper distribution → less randomness

T = 1  → original distribution

T > 1  → flatter distribution → more randomness
```

Temperature therefore acts as a simple control over how strongly generation favors the model's highest-scoring token candidates.


In [16]:
#top k sampling
#Temperature changes the shape of the whole distribution. but top k completely removes all but the top k token candidates.

k = 5

top_k_values, top_k_indices = torch.topk(last_token_logits,k=k)

threshold = top_k_values[:, -1:]
filtered_logits = last_token_logits.clone()
filtered_logits[filtered_logits < threshold] = float("-inf")
top_k_probs = torch.softmax(filtered_logits,dim=-1)
top_probs, top_indices = torch.topk(top_k_probs,k=10)

for prob, token_id in zip(top_probs[0], top_indices[0]):
    token = tokenizer.decode([token_id.item()])

    print(
        f"{repr(token):12s} | "
        f"{prob.item() * 100:.2f}%"
    )

    #as u can see , after 5 everything is zero because softmax of minus inf is always 0

' the'       | 60.84%
' a'         | 25.78%
' likely'    | 5.66%
' probably'  | 4.77%
' __'        | 2.94%
'$'          | 0.00%
'%'          | 0.00%
'#'          | 0.00%
'!'          | 0.00%
'"'          | 0.00%


## Top-k Sampling

`k = 5`

We choose how many token candidates we want to keep. With `k = 5`, only the 5 highest-scoring tokens will be allowed to have a non-zero probability.

---

`top_k_values, top_k_indices = torch.topk(last_token_logits, k=k)`

`last_token_logits` currently has shape:

```text
[1, 151936]
```

This means we have one request and 151,936 possible next-token logits.

`torch.topk()` finds the `k` largest values.

It returns **two results**, which Python unpacks into two variables:

```text
top_k_values   → the actual top 5 logit values
top_k_indices  → where those values were located
```

Since those positions correspond to the vocabulary, `top_k_indices` are also the token IDs.

After this:

```text
top_k_values.shape  = [1, 5]
top_k_indices.shape = [1, 5]
```

---

`threshold = top_k_values[:, -1:]`

`top_k_values` contains the 5 largest logits ordered from highest to lowest.

So the final value is the **5th-highest logit**.

We use that as our cutoff.

Here:

```text
:
```

means keep every item in the batch.

And:

```text
-1:
```

means start from the final value and keep everything from there.

Using `-1:` instead of just `-1` keeps the dimension.

So:

```text
[1, 5] → [1, 1]
```

Now `threshold` contains only the 5th-highest logit.

Any token with a logit below this value should be removed.

---

`filtered_logits = last_token_logits.clone()`

We make a copy of the original logits.

`.clone()` is important because we are about to modify this tensor.

Without the clone, we would modify `last_token_logits` itself and lose the original model output.

Initially:

```text
last_token_logits → [1, 151936]
filtered_logits   → [1, 151936]
```

Both contain the same values, but they are separate tensors.

---

`filtered_logits[filtered_logits < threshold] = float("-inf")`

This is the actual filtering step.

First:

```text
filtered_logits < threshold
```

checks every vocabulary logit against the cutoff.

For example:

```text
15.47 < 12.44 → False
14.61 < 12.44 → False
13.10 < 12.44 → False
12.93 < 12.44 → False
12.44 < 12.44 → False
12.43 < 12.44 → True
```

This produces a boolean mask containing `True` wherever a token should be removed.

PyTorch syntax like:

```text
tensor[condition] = value
```

means:

> replace every value where the condition is `True`.

So this line means:

```text
if logit < top-k threshold:
    replace it with -inf
```

The top 5 logits stay unchanged, while everything below them becomes `-inf`.

The tensor shape is still:

```text
[1, 151936]
```

We did not remove tensor elements. We only changed their values.

---

`top_k_probs = torch.softmax(filtered_logits, dim=-1)`

Now we convert the filtered logits into probabilities.

`dim=-1` means apply softmax across the **last dimension**.

Our shape is:

```text
[1, 151936]
```

so the last dimension is the vocabulary dimension containing the 151,936 candidate tokens.

The filtered tokens currently have:

```text
logit = -inf
```

and softmax effectively gives them:

```text
exp(-inf) = 0
```

so their probability becomes exactly `0`.

Only the top 5 tokens remain with non-zero probability.

Softmax also renormalizes those surviving probabilities so that together they sum to approximately 1.

---

`top_probs, top_indices = torch.topk(top_k_probs, k=10)`

Now we inspect the 10 largest probabilities.

Again, `torch.topk()` returns two things:

```text
top_probs    → probability values
top_indices  → token IDs where those probabilities occur
```

We intentionally ask for 10 even though only 5 tokens survived.

This lets us verify that:

```text
first 5 tokens → non-zero probability
next 5 tokens  → 0 probability
```

---

`for prob, token_id in zip(top_probs[0], top_indices[0]):`

Both tensors have a batch dimension.

Their shape is:

```text
[1, 10]
```

`[0]` selects the first request in the batch, giving us 10 probability values and 10 corresponding token IDs.

`zip()` pairs them together:

```text
probability 1 ↔ token ID 1
probability 2 ↔ token ID 2
...
```

so we can process each candidate one at a time.

---

`token = tokenizer.decode([token_id.item()])`

`token_id` is still a PyTorch tensor.

`.item()` extracts the normal Python integer from that tensor.

For example:

```text
tensor(279) → 279
```

We place it inside a list because `tokenizer.decode()` expects one or more token IDs:

```text
[279]
```

The tokenizer then converts that token ID back into readable text:

```text
279 → " the"
```

---

The complete top-k process is therefore:

```text
151,936 logits
      ↓
find the top k values
      ↓
use the kth value as the cutoff
      ↓
everything below the cutoff → -inf
      ↓
softmax
      ↓
filtered tokens → probability 0
      ↓
sample only from the surviving top-k tokens
```


In [17]:
#top p sampling
#using cf

p = 0.9

sorted_probs, sorted_indices = torch.sort(probs, descending=True)

cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

remove_mask = cumulative_probs > p
remove_mask[:, 1:] = remove_mask[:, :-1].clone()
remove_mask[:, 0] = False

sorted_probs[remove_mask] = 0.0
sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

top_probs, top_positions = torch.topk(sorted_probs, k=10)

for prob, position in zip(top_probs[0], top_positions[0]):
    token_id = sorted_indices[0, position]
    token = tokenizer.decode([token_id.item()])

    print(
        f"{repr(token):12s} | "
        f"{prob.item() * 100:.2f}%"
    )

' the'       | 47.97%
' a'         | 20.31%
' likely'    | 4.46%
' probably'  | 3.76%
' __'        | 2.32%
' not'       | 2.30%
':\n'        | 1.89%
' ______'    | 1.84%
' an'        | 1.67%
' also'      | 1.54%


## Top-p Sampling

Top-k always keeps a fixed number of tokens. Top-p is dynamic: it keeps **as many of the highest-probability tokens as needed to reach a chosen amount of total probability mass**.

For `p = 0.9`, we want the smallest group of likely tokens that together account for roughly 90% of the model's probability distribution.

---

`p = 0.9`

This sets our probability threshold to 90%.

Unlike `k = 5`, this does **not** mean we will keep a fixed number of tokens. One decoding step might need 8 tokens to reach 90%, while another might need 50.

It depends entirely on how spread out the model's current distribution is.

---

`sorted_probs, sorted_indices = torch.sort(probs, descending=True)`

Before sorting:

```text
probs.shape = [1, 151936]
```

`probs` contains the probability of every vocabulary token, but those probabilities are stored according to token ID, not from highest to lowest probability.

Top-p needs to start with the most likely token and keep adding probabilities until we reach `p`, so we first sort them.

`descending=True` means:

> sort from largest probability to smallest probability.

`torch.sort()` returns two things:

```text
sorted_probs
→ the probabilities after sorting

sorted_indices
→ where each probability originally came from
```

For example, imagine:

```text
original probabilities:

token 0 → 0.05
token 1 → 0.40
token 2 → 0.10
token 3 → 0.30
```

After sorting:

```text
sorted_probs:

0.40
0.30
0.10
0.05
```

and:

```text
sorted_indices:

1
3
2
0
```

The indices matter because after sorting, position `0` no longer means token ID `0`.

It now means:

> the highest-probability token.

`sorted_indices` lets us later recover the actual token ID.

The shape stays:

```text
[1, 151936]
```

because we only changed the order of the elements.

---

`cumulative_probs = torch.cumsum(sorted_probs, dim=-1)`

`cumsum` means **cumulative sum**.

Instead of looking at each probability separately, it keeps a running total.

Suppose our sorted probabilities were:

```text
0.43
0.18
0.10
0.07
0.05
```

The cumulative probabilities would become:

```text
0.43             → 0.43
0.43 + 0.18      → 0.61
0.61 + 0.10      → 0.71
0.71 + 0.07      → 0.78
0.78 + 0.05      → 0.83
```

So each position tells us:

> how much total probability have we collected if we keep every token up to this point?

`dim=-1` tells PyTorch to perform this running sum across the final dimension.

Our tensor is:

```text
[1, 151936]
     ↑
 vocabulary
```

so we want the running sum across the vocabulary probabilities.

The shape remains:

```text
[1, 151936]
```

---

`remove_mask = cumulative_probs > p`

Now we compare every cumulative probability with our threshold:

```text
p = 0.9
```

For example:

```text
0.43 > 0.9 → False
0.61 > 0.9 → False
0.78 > 0.9 → False
0.89 > 0.9 → False
0.93 > 0.9 → True
0.96 > 0.9 → True
```

This creates a boolean mask.

Conceptually:

```text
False
False
False
False
True
True
...
```

`True` means:

> this position is past our top-p boundary and should eventually be removed.

But there is one important problem with this mask.

---

`remove_mask[:, 1:] = remove_mask[:, :-1].clone()`

Suppose the cumulative probabilities are:

```text
0.82
0.88
0.92
0.95
```

With:

```text
p = 0.9
```

our first mask would be:

```text
False
False
True
True
```

The token that brought us from:

```text
0.88 → 0.92
```

is currently marked for removal.

But that token is the token that actually allowed us to **reach 90%**.

If we remove it, the remaining tokens only contain 88% probability mass.

So we shift the mask one position to the right.

Before:

```text
False
False
True
True
```

After shifting:

```text
?
False
False
True
```

Now the token that crossed the threshold is kept, and removal begins with the token after it.

### What do `[:, 1:]` and `[:, :-1]` mean?

Our mask has shape:

```text
[1, 151936]
```

In:

```text
[:, 1:]
```

the first `:` means:

> keep every batch item.

`1:` means:

> start from position 1 and continue to the end.

So it selects every position except the first one.

In:

```text
[:, :-1]
```

again, `:` keeps every batch item.

`:-1` means:

> start from the beginning and stop before the final position.

So:

```text
[:, 1:]     → positions 1 → end
[:, :-1]    → positions 0 → second-last
```

Assigning one to the other effectively shifts the mask one position to the right.

### Why `.clone()`?

The source and destination are slices of the **same tensor**.

We are modifying `remove_mask` while also reading values from `remove_mask`.

`.clone()` first creates a separate copy of the original source values, so our assignment cannot accidentally overwrite values that we still need while performing the shift.

---

`remove_mask[:, 0] = False`

After shifting the mask, the first position does not receive a previous value because there is nothing before position 0.

So we explicitly set it to:

```text
False
```

This also guarantees that the highest-probability token is always kept.

Even in an extreme case where the first token alone already has more probability than `p`, top-p must still keep at least one token.

---

`sorted_probs[remove_mask] = 0.0`

Now we apply the boolean mask.

This uses the same PyTorch pattern as top-k:

```text
tensor[condition] = value
```

Every position where `remove_mask` is `True` gets replaced with:

```text
0.0
```

So:

```text
tokens inside the nucleus  → keep their probability
tokens outside the nucleus → probability = 0
```

At this point, tokens outside our top-p candidate set can no longer be sampled.

The tensor shape is still:

```text
[1, 151936]
```

We are changing values, not deleting tensor positions.

---

`sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)`

After filtering, the surviving probabilities no longer sum to 1.

For example, we might have kept:

```text
0.43
0.18
0.10
0.08
0.06
0.05
```

which together might sum to something like:

```text
0.90
```

Renormalization makes the surviving values an explicit probability distribution that sums to 1.

`torch.multinomial()` can also sample from non-negative, unnormalized weights, so this normalization is not required by PyTorch. Here we divide every surviving probability by their total to make the filtered distribution explicit and easy to inspect.

Conceptually:

```text
new_probability = old_probability / surviving_probability_sum
```

This makes the remaining probabilities sum to 1 again.

That process is called **renormalization**.

This is why the highest token in our experiment increased from its original probability of about 43% to roughly 48% after top-p filtering: some lower-probability tokens were removed, and the surviving probability mass was rescaled back to 100%.

### Why `dim=-1`?

The current shape is still:

```text
[1, 151936]
```

We want to add together all token probabilities for each request, so we sum across the final vocabulary dimension.

### Why `keepdim=True`?

Without `keepdim=True`:

```text
[1, 151936]
      ↓ sum
[1]
```

With `keepdim=True`:

```text
[1, 151936]
      ↓ sum
[1, 1]
```

Keeping the dimension makes division straightforward:

```text
[1, 151936] / [1, 1]
```

PyTorch broadcasts that one total across all 151,936 probabilities.

---

`top_probs, top_positions = torch.topk(sorted_probs, k=10)`

At this point, `sorted_probs` is already sorted from highest to lowest, but we use `torch.topk()` here simply to inspect the strongest surviving probabilities.

There is an important naming detail here:

```text
top_positions
```

contains positions inside the **sorted tensor**.

They are **not the original vocabulary token IDs**.

This is different from earlier, where we ran `torch.topk()` directly on a vocabulary-aligned tensor and its indices could directly represent token IDs.

Because we sorted the vocabulary first, we need one more lookup.

---

`token_id = sorted_indices[0, position]`

`sorted_indices` remembers which original vocabulary token ended up at each sorted position.

So if:

```text
position = 0
```

that means:

> give me the token at the first position of our sorted distribution.

Then:

```text
sorted_indices[0, 0]
```

might return:

```text
279
```

which is the real vocabulary token ID.

So the mapping is:

```text
sorted position
      ↓
sorted_indices
      ↓
original token ID
      ↓
tokenizer.decode()
      ↓
text
```

This mapping is necessary because sorting changed the order of the vocabulary.

---

The main difference between top-k and top-p is therefore:

```text
Top-k:
keep a fixed number of tokens

Top-p:
keep a dynamic number of tokens
until enough probability mass has been collected
```

With top-p, the size of the candidate set adapts to how confident the model is.

If the distribution is very concentrated:

```text
Token A → 70%
Token B → 15%
Token C → 7%
```

only a few tokens may be needed to reach `p = 0.9`.

If the distribution is much flatter:

```text
Token A → 8%
Token B → 7%
Token C → 6%
Token D → 5%
...
```

many more tokens may be required.

That adaptive candidate set is the main idea behind nucleus sampling.


In [18]:
#using actual sampling code (temp, top k and then top p)

def sample_next_token(last_token_logits, temperature=1.0, top_k=50, top_p=0.9):

    # 1. temperature
    logits = last_token_logits / temperature

    # 2. top-k filtering
    top_k_values, _ = torch.topk(logits, k=top_k)
    threshold = top_k_values[:, -1:]
    logits = logits.clone()
    logits[logits < threshold] = float("-inf")

    # 3. convert logits to probabilities
    probs = torch.softmax(logits, dim=-1)

    # 4. top-p filtering
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    remove_mask = cumulative_probs > top_p
    remove_mask[:, 1:] = remove_mask[:, :-1].clone()
    remove_mask[:, 0] = False

    sorted_probs[remove_mask] = 0.0

    # 5. renormalize
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)

    # 6. sample one position from the filtered distribution
    sampled_position = torch.multinomial(sorted_probs, num_samples=1)

    # 7. convert sorted position back to original token ID
    next_token = torch.gather(sorted_indices, dim=-1, index=sampled_position)

    return next_token

In [19]:
#validation


for _ in range(10):
    next_token = sample_next_token(
        last_token_logits,
        temperature=1.0,
        top_k=50,
        top_p=0.9
    )

    print(
        next_token.item(),
        repr(tokenizer.decode(next_token[0]))
    )

4363 ' likely'
279 ' the'
279 ' the'
279 ' the'
279 ' the'
1304 ' __'
279 ' the'
279 ' the'
264 ' a'
279 ' the'


In [21]:
# putting everything into a naive autoregressive decode loop

prompt = "The fastest animal in the world is"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]

generated_ids = input_ids.clone()

for _ in range(30): #generating next 30 toks

    with torch.no_grad():
        outputs = model(generated_ids)

    last_token_logits = outputs.logits[:, -1, :]

    next_token = sample_next_token(
        last_token_logits,
        temperature=0.8,
        top_k=50,
        top_p=0.9
    )

    generated_ids = torch.cat(
        [generated_ids, next_token],
        dim=-1
    )

    if next_token.item() == tokenizer.eos_token_id:
        break

print(tokenizer.decode(generated_ids[0]))

The fastest animal in the world is a certain species of fish. This species of fish can swim at a speed of 100 kilometers per hour. If this fish swims for 


## Putting Sampling Into a Naive Autoregressive Decode Loop

Now that temperature, top-k, top-p, and multinomial sampling work independently, we can plug them into a naive autoregressive decode loop.

`prompt = "The fastest animal in the world is"`

This is the text we want the model to continue.

---

`inputs = tokenizer(prompt, return_tensors="pt").to(model.device)`

The tokenizer converts the prompt into token IDs.

`return_tensors="pt"` tells the tokenizer to return PyTorch tensors.

`.to(model.device)` moves those tensors to the same GPU as the model.

---

`input_ids = inputs["input_ids"]`

This extracts the actual token ID tensor.

For our prompt, its shape is:

```text
[1, 7]
```

which means:

```text
1 request
7 prompt tokens
```

---

`generated_ids = input_ids.clone()`

We create a copy of the original prompt tokens.

`generated_ids` will hold the complete sequence:

```text
original prompt tokens
+
every token we generate afterward
```

We use `.clone()` so the original `input_ids` stays unchanged.

At the beginning:

```text
generated_ids.shape = [1, 7]
```

---

`for _ in range(30):`

We allow the model to generate at most 30 new tokens.

The `_` means we do not actually care about the current loop number. We only want the loop to run repeatedly.

The loop can also stop earlier if the model generates its EOS token.

---

`with torch.no_grad():`

We are doing inference, not training.

---

`outputs = model(generated_ids)`

We pass the sequence generated so far into the model.

On the first decoding step:

```text
generated_ids.shape = [1, 7]
```

After generating one token:

```text
[1, 8]
```

then:

```text
[1, 9]
```

and so on.

In this simple version, we pass the entire sequence back through the model every time.

This is intentionally inefficient. A real inference engine would use the KV cache so previous tokens do not need to be recomputed on every decode step.

---

`last_token_logits = outputs.logits[:, -1, :]`

The model produces logits for every sequence position.

If the current sequence contains 7 tokens:

```text
outputs.logits.shape = [1, 7, 151936]
```

The dimensions are:

```text
batch
sequence positions
vocabulary logits
```

We only need the logits from the final sequence position because those logits predict the **next token**.

In:

```text
[:, -1, :]
```

the first `:` keeps every batch item.

`-1` selects the final sequence position.

The last `:` keeps every vocabulary logit.

So:

```text
[1, 7, 151936]
        ↓
[1, 151936]
```

Now we have one score for every possible next token.

---

`next_token = sample_next_token(...)`

Instead of using `argmax`, we pass the logits through the sampling function we built.

Inside that function, the sequence is:

```text
logits
  ↓
temperature
  ↓
top-k filtering
  ↓
softmax
  ↓
top-p filtering
  ↓
renormalization
  ↓
multinomial sampling
  ↓
next token ID
```

In this run we used:

```text
temperature = 0.8
top_k = 50
top_p = 0.9
```

The returned token has shape:

```text
[1, 1]
```

meaning one sampled next token for one request.

---

`generated_ids = torch.cat([generated_ids, next_token], dim=-1)`

Now we append the sampled token to the sequence.

`torch.cat()` joins tensors together.

Suppose:

```text
generated_ids.shape = [1, 7]
next_token.shape    = [1, 1]
```

`dim=-1` means concatenate along the last dimension, which here is the token sequence dimension.

So:

```text
[1, 7] + [1, 1]
        ↓
[1, 8]
```

After another decoding step:

```text
[1, 8] + [1, 1]
        ↓
[1, 9]
```

This is how autoregressive generation grows one token at a time.

---

`if next_token.item() == tokenizer.eos_token_id:`

The tokenizer has a special **EOS**, or end-of-sequence, token.

If the model samples this token, it is signalling that generation should stop.

`.item()` converts the one-value PyTorch tensor into a normal Python integer so we can compare it with the EOS token ID.

---

`break`

If EOS was generated, `break` exits the decoding loop immediately.

Otherwise, generation continues until either EOS appears or the 30-token limit is reached.

---

`tokenizer.decode(generated_ids[0])`

At the end, `generated_ids` contains both:

```text
prompt tokens + generated tokens
```

`generated_ids[0]` selects the first request in the batch.

The tokenizer then converts all of those token IDs back into readable text.

The important result here is not whether the generated statement is factually correct. Sampling controls **which tokens are selected from the model's distribution**; it does not guarantee factual accuracy.

At this point we have built the full sampling path ourselves:

```text
model forward pass
        ↓
next-token logits
        ↓
temperature
        ↓
top-k
        ↓
softmax
        ↓
top-p
        ↓
renormalize
        ↓
multinomial sampling
        ↓
append sampled token
        ↓
repeat
```

This completes **Chapter 2: Sampling**.

We now have a naive autoregressive decode loop that can generate tokens using a real sampling pipeline instead of always relying on greedy `argmax` decoding.


## Bonus: Why Do We Need Temperature, Top-k and Top-p?

At every decoding step, the model gives us a distribution over possible next tokens.

For example:

```text
"the"       → 43%
"a"         → 18%
"likely"    → 4%
"probably"  → 3%
...
```

The model itself does not decide:

> "Use argmax."

or:

> "Use top-p with temperature 0.8."

It only gives us the scores.

The **decoding strategy** decides how we use those scores.

This matters because there are two bad extremes.

### Extreme 1: Always use argmax

With argmax we simply choose:

```text
highest probability token
```

every single step.

So if:

```text
"the" → 43%
"a"   → 18%
```

argmax always chooses:

```text
"the"
```

The remaining distribution is completely ignored.

Argmax is therefore:

```text
logits
  ↓
find maximum
  ↓
choose it
```

There is no randomness.

This can be useful when we want highly repeatable output, but repeatedly taking the locally most likely token can also make open-ended generation less diverse and sometimes repetitive or bland. This failure mode was one of the motivations behind nucleus sampling research.

---

## Sampling Without Any Controls Has the Opposite Problem

Suppose we instead do:

```text
logits
  ↓
softmax
  ↓
sample from all 151,936 tokens
```

Now even extremely unlikely tokens technically have some chance of being selected.

Imagine:

```text
"the"       → 43%
"a"         → 18%
"likely"    → 4%

...

some strange token → 0.00001%
```

The strange token is incredibly unlikely, but its probability is still not zero.

Across long generations and millions of requests, sampling directly from the entire tail can occasionally select low-quality tokens.

So we have two problems:

```text
argmax
→ too restrictive

unrestricted sampling
→ can sample from the unreliable tail
```

Sampling controls exist largely to operate somewhere between those extremes.

---

# Temperature: Control How Strongly We Trust the Ranking

Temperature changes the **shape of the probability distribution**.

It does not directly remove tokens.

We apply:

```text
logit / temperature
```

before softmax.

### Lower temperature

```text
T < 1
```

makes high-scoring tokens dominate more strongly.

For our experiment:

```text
T = 1.0

"the" → 43.24%
```

but:

```text
T = 0.5

"the" → 82.81%
```

The model becomes much more confident in its strongest candidates.

So lower temperature means:

```text
distribution becomes sharper
        ↓
high-ranked tokens dominate
        ↓
less randomness
```

This is useful when we care more about consistency than diversity.

Examples can include tasks such as:

```text
information extraction
structured responses
classification-like generation
some coding/evaluation workloads
```

where we generally do not want wildly different answers every time.

---

### Higher temperature

With:

```text
T > 1
```

the probability distribution becomes flatter.

In our experiment:

```text
T = 1.5

"the" → 10.75%
```

instead of 43%.

Probability mass gets spread across many more tokens.

So:

```text
higher temperature
        ↓
tokens become more competitive
        ↓
more possible continuations
        ↓
more randomness/diversity
```

This can be useful for more open-ended generation such as brainstorming or creative writing.

Temperature therefore answers:

> **How strongly should we prefer the model's highest-ranked tokens?**

Serving systems such as vLLM expose temperature precisely as a randomness control; temperature `0` corresponds to greedy sampling in its API.

---

# Top-k: Put a Hard Limit on the Candidate Set

Top-k answers a different question:

> **How many tokens are even allowed to compete?**

If:

```text
k = 5
```

we keep only:

```text
top 5 tokens
```

and everything else gets:

```text
probability = 0
```

So even if the vocabulary contains:

```text
151,936 tokens
```

our sampler can only choose from 5.

Conceptually:

```text
151,936 candidates
       ↓
keep best 5
       ↓
sample from those 5
```

This protects sampling from reaching deep into the low-probability tail.

Temperature cannot do this.

A low temperature may make an unlikely token **extremely unlikely**, but its probability can still remain above zero.

Top-k says:

> No. This token cannot be sampled at all.

That is its independent contribution.

---

## The Limitation of Top-k

The problem is that a fixed `k` does not understand how confident the model currently is.

Imagine one decoding step:

```text
token A → 80%
token B → 10%
token C → 5%
```

The model is extremely confident.

Keeping 50 tokens here may be unnecessary.

But on another decoding step:

```text
token A → 5%
token B → 4.8%
token C → 4.6%
token D → 4.4%
...
```

the distribution is much flatter.

Now limiting ourselves to only a few tokens may remove many perfectly reasonable choices.

So:

```text
top-k = fixed candidate count
```

regardless of the shape of the current distribution. Hugging Face and vLLM both define top-k as retaining a fixed number of the highest-probability vocabulary candidates.

---

# Top-p: Make the Candidate Set Adapt to the Model

Top-p solves this differently.

Instead of saying:

```text
keep 50 tokens
```

we say:

```text
keep enough tokens to cover 90% probability
```

for:

```text
top_p = 0.9
```

This makes the candidate count dynamic.

### When the model is confident

Suppose:

```text
A → 60%
B → 20%
C → 11%
```

Cumulative probability:

```text
A         → 60%
A + B     → 80%
A + B + C → 91%
```

Only 3 tokens may be needed.

### When the model is uncertain

Suppose:

```text
A → 10%
B → 9%
C → 8%
D → 7%
...
```

We may need dozens of tokens before reaching 90%.

So top-p automatically adapts:

```text
confident distribution
→ small candidate set

uncertain distribution
→ larger candidate set
```

This is the key difference from top-k.

Top-p was introduced as **nucleus sampling**: sample from a dynamic high-probability region while cutting off the unreliable tail.

So top-p answers:

> **How much of the model's probability mass should be allowed to participate?**

---

# What Each Control Contributes Independently

The easiest way to separate them is:

```text
Temperature
→ changes the probabilities

Top-k
→ limits the number of candidates

Top-p
→ limits the amount of probability mass considered

Multinomial
→ actually performs the random choice
```


